# Phase 0 — Raw checkpoint reload verification
Uses Hùng stage-1 epoch-2 checkpoint only. No training or model selection. Outputs remain on Kaggle.


In [1]:
"""Standalone phase-0 Kaggle verifier for Hùng's selected epoch-2 checkpoint.

Paste this entire source into a new Kaggle notebook cell with the stage-1 notebook
output attached as input. The source and output paths are fixed deliberately.
"""

import csv
import hashlib
import json
import math
import platform
import time
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer


RUN_ID = "phase0_hung_raw_verify_20261121_01"
STAGE1 = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique/fg_full_stage1")
DATA = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique/fg_full_data/deberta_fg_dev.csv")
MODEL = STAGE1 / "best_model_after_2_epochs"
OUT = Path("/kaggle/working") / RUN_ID
EXPECTED_DEV_SHA256 = "F7EC0B2308EBDC811BFC11828A076F498FD2CB542C503274DAD5CBA852EBD1BC"
EXPECTED_STAGE1_F1 = 0.7935989549314174
EXPECTED_ROWS = 6404
BATCH_SIZE = 8
THRESHOLD = 0.5


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


def metrics_from_pairs(labels, predictions):
    if len(labels) != EXPECTED_ROWS or len(predictions) != EXPECTED_ROWS:
        raise ValueError("expected exactly 6,404 paired rows")
    tp = sum(y == 1 and p == 1 for y, p in zip(labels, predictions))
    fp = sum(y == 0 and p == 1 for y, p in zip(labels, predictions))
    fn = sum(y == 1 and p == 0 for y, p in zip(labels, predictions))
    tn = EXPECTED_ROWS - tp - fp - fn
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "rows": EXPECTED_ROWS, "positive_class": 1, "threshold": THRESHOLD,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "confusion_matrix_labels_0_1": [[tn, fp], [fn, tp]],
        "precision": precision, "recall": recall, "f1": f1,
        "accuracy": (tp + tn) / EXPECTED_ROWS,
        "flag_rate_prediction_0": (tn + fn) / EXPECTED_ROWS,
    }


def main():
    required = [
        STAGE1 / "stage1_results.json", DATA,
        MODEL / "config.json", MODEL / "model.safetensors",
        MODEL / "tokenizer.json", MODEL / "tokenizer_config.json",
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError("missing stage-1 output: " + "; ".join(missing))
    report = json.loads((STAGE1 / "stage1_results.json").read_text(encoding="utf-8"))
    config = json.loads((MODEL / "config.json").read_text(encoding="utf-8"))
    if report.get("completed_epochs") != 2 or report.get("dev_examples") != EXPECTED_ROWS:
        raise ValueError("stage-1 report is not the 2-epoch full MuSiQue run")
    expected = report.get("best_dev_metrics", {})
    if not math.isclose(expected.get("eval_f1", -1), EXPECTED_STAGE1_F1, abs_tol=1e-12):
        raise ValueError("unexpected stage-1 selected F1")
    if config.get("label2id", {}).get("ANSWERABLE") != 1 or config.get("label2id", {}).get("NOT-ANSWERABLE") != 0:
        raise ValueError("checkpoint label mapping is unexpected")
    dev_hash = sha256_file(DATA)
    if dev_hash != EXPECTED_DEV_SHA256:
        raise ValueError(f"dev CSV SHA256 mismatch: {dev_hash}")

    texts, labels, text_hashes = [], [], []
    with DATA.open("r", encoding="utf-8-sig", newline="") as source:
        reader = csv.DictReader(source, restkey="_extra")
        if reader.fieldnames != ["text", "label"]:
            raise ValueError("dev header must be exactly text,label")
        for index, row in enumerate(reader):
            if set(row) != {"text", "label"} or row["label"] not in ("0", "1"):
                raise ValueError(f"invalid CSV row {index}")
            text = row["text"]
            if not text.startswith("Sub-question: ") or "\n\nPassages:\n" not in text:
                raise ValueError(f"invalid formatter at row {index}")
            texts.append(text)
            labels.append(int(row["label"]))
            text_hashes.append(hashlib.sha256(text.encode("utf-8")).hexdigest())
    if len(texts) != EXPECTED_ROWS or labels.count(1) != 3145 or labels.count(0) != 3259:
        raise ValueError("dev row count or label distribution mismatch")
    if sha256_file(DATA) != dev_hash:
        raise ValueError("dev CSV changed during audit")

    OUT.mkdir(parents=True, exist_ok=False)
    log_path = OUT / "verify.log"

    def log(event, **fields):
        line = json.dumps({"event": event, "time_utc": __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat(), **fields})
        with log_path.open("a", encoding="utf-8") as destination:
            destination.write(line + "\n")
        print(line, flush=True)

    started = time.monotonic()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    log("start", run_id=RUN_ID, device=device, rows=len(texts), checkpoint=str(MODEL))
    file_hashes = {str(path.relative_to(STAGE1.parent)): sha256_file(path) for path in required}
    tokenizer = AutoTokenizer.from_pretrained(MODEL, local_files_only=True, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, local_files_only=True).to(device).eval()
    if model.config.label2id.get("ANSWERABLE") != 1 or model.config.label2id.get("NOT-ANSWERABLE") != 0:
        raise ValueError("loaded model label mapping is unexpected")

    predictions_path = OUT / "predictions.csv"
    with predictions_path.open("x", encoding="utf-8", newline="") as destination:
        writer = csv.writer(destination)
        writer.writerow(["row_index", "text_sha256", "label", "prob_answerable", "prediction"])
        for start in range(0, EXPECTED_ROWS, BATCH_SIZE):
            batch = texts[start:start + BATCH_SIZE]
            inputs = tokenizer(batch, max_length=512, padding=True, truncation=True,
                               pad_to_multiple_of=8 if device == "cuda" else None,
                               return_tensors="pt").to(device)
            with torch.inference_mode():
                probabilities = torch.softmax(model(**inputs).logits.float(), dim=-1)[:, 1].cpu().tolist()
            for offset, probability in enumerate(probabilities):
                index = start + offset
                writer.writerow([index, text_hashes[index], labels[index], repr(probability), int(probability >= THRESHOLD)])
            if (start // BATCH_SIZE + 1) % 50 == 0 or start + len(batch) == EXPECTED_ROWS:
                destination.flush()
                log("progress", completed_rows=start + len(batch), total_rows=EXPECTED_ROWS,
                    elapsed_seconds=round(time.monotonic() - started, 1))

    readback_predictions = []
    with predictions_path.open("r", encoding="utf-8", newline="") as source:
        reader = csv.DictReader(source)
        if reader.fieldnames != ["row_index", "text_sha256", "label", "prob_answerable", "prediction"]:
            raise ValueError("prediction header mismatch")
        for index, row in enumerate(reader):
            if index >= EXPECTED_ROWS or int(row["row_index"]) != index:
                raise ValueError(f"missing, extra, or out-of-order prediction at {index}")
            if row["text_sha256"] != text_hashes[index] or int(row["label"]) != labels[index]:
                raise ValueError(f"source mismatch at prediction row {index}")
            probability = float(row["prob_answerable"])
            prediction = int(row["prediction"])
            if not math.isfinite(probability) or not 0 <= probability <= 1 or prediction != int(probability >= THRESHOLD):
                raise ValueError(f"invalid prediction at row {index}")
            readback_predictions.append(prediction)
    if len(readback_predictions) != EXPECTED_ROWS:
        raise ValueError("incomplete predictions.csv")

    metrics = metrics_from_pairs(labels, readback_predictions)
    metrics["reported_comparison"] = {
        name: {"recomputed": metrics[name], "reported": expected["eval_" + name],
               "absolute_difference": abs(metrics[name] - expected["eval_" + name])}
        for name in ("precision", "recall", "f1", "accuracy")
    }
    metrics["report_match_within_1e-4"] = all(
        value["absolute_difference"] <= 1e-4 for value in metrics["reported_comparison"].values()
    )
    manifest = {
        "run_id": RUN_ID,
        "status": "completed" if metrics["report_match_within_1e-4"] else "report_mismatch",
        "source_notebook": "https://www.kaggle.com/code/hungluongduc/fact-grounding-gap-deberta-full-musique",
        "source_run_id": "371432426",
        "selected_checkpoint": str(MODEL),
        "source_dev_sha256": dev_hash,
        "source_text_hash_method": "SHA256 of each parsed text encoded as UTF-8, row_index 0-based",
        "input_file_sha256": file_hashes,
        "predictions_sha256": sha256_file(predictions_path),
        "runtime": {"python": platform.python_version(), "torch": torch.__version__,
                    "transformers": transformers.__version__, "device": device,
                    "batch_size": BATCH_SIZE, "elapsed_seconds": time.monotonic() - started},
    }
    (OUT / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
    (OUT / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    log("complete", status=manifest["status"], rows=EXPECTED_ROWS, f1=metrics["f1"])
    print(json.dumps({"output_dir": str(OUT), "status": manifest["status"], "metrics": metrics}, indent=2))
    if not metrics["report_match_within_1e-4"]:
        raise ValueError("recomputed metrics do not match stage-1 report")


if __name__ == "__main__":
    main()


{"event": "start", "time_utc": "2026-11-21T04:47:16.865428+00:00", "run_id": "phase0_hung_raw_verify_20261121_01", "device": "cuda", "rows": 6404, "checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique/fg_full_stage1/best_model_after_2_epochs"}


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

{"event": "progress", "time_utc": "2026-11-21T04:48:10.977963+00:00", "completed_rows": 400, "total_rows": 6404, "elapsed_seconds": 54.4}
{"event": "progress", "time_utc": "2026-11-21T04:48:52.121335+00:00", "completed_rows": 800, "total_rows": 6404, "elapsed_seconds": 95.5}
{"event": "progress", "time_utc": "2026-11-21T04:49:38.913984+00:00", "completed_rows": 1200, "total_rows": 6404, "elapsed_seconds": 142.3}
{"event": "progress", "time_utc": "2026-11-21T04:50:23.198029+00:00", "completed_rows": 1600, "total_rows": 6404, "elapsed_seconds": 186.6}
{"event": "progress", "time_utc": "2026-11-21T04:51:07.834610+00:00", "completed_rows": 2000, "total_rows": 6404, "elapsed_seconds": 231.2}
{"event": "progress", "time_utc": "2026-11-21T04:51:52.117283+00:00", "completed_rows": 2400, "total_rows": 6404, "elapsed_seconds": 275.5}
{"event": "progress", "time_utc": "2026-11-21T04:52:37.341908+00:00", "completed_rows": 2800, "total_rows": 6404, "elapsed_seconds": 320.7}
{"event": "progress", "t